In [13]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd


pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)


# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_test_fe.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [20]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [ ]:
# ============================================================
# 3. 최종모델(15) + 데이터셋 필터링
# ============================================================
# wl_diff_6h < -n 인 train, val 제거하고 진행

train_filtered = train.loc[
    train['wl_diff_6h'] >= -1
].copy()

val_filtered = val.loc[
    val['wl_diff_6h'] >= -1
].copy()

print('Train:', train.shape, '->', train_filtered.shape)
print('Validation:', val.shape, '->', val_filtered.shape)

print('Train 제거 행:', len(train) - len(train_filtered))
print('Validation 제거 행:', len(val) - len(val_filtered))

# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train_filtered[target_col]
y_val = val_filtered[target_col]


# station_id가 포함된 데이터셋 생성
# 기존 데이터와의 연결 차단을 위해 .copy() 사용
X_train_with_station = train_filtered[num_cols_fe + ['station_id']].copy()
X_val_with_station = val_filtered[num_cols_fe + ['station_id']].copy()

# CatBoost 전용 범주형 변수 취급: str로 자료형 변환
X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_val_with_station['station_id'] = X_val_with_station['station_id'].astype(str)

# 모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train,
    early_stopping_rounds=100,
    eval_set=(X_val_with_station, y_val),
    verbose=200
)

# 예측
pred_val_no_ohe = model_base_no_ohe.predict(X_val_with_station)
pred_train_no_ohe = model_base_no_ohe.predict(X_train_with_station)

print('Best iteration:', model_base_no_ohe.best_iteration_)
print('Tree count:', model_base_no_ohe.tree_count_)
print('Best score:', model_base_no_ohe.best_score_['validation']['RMSE'])
print()

rmse_val_no_ohe = root_mean_squared_error(
    y_val, pred_val_no_ohe
)

rmse_train_no_ohe = root_mean_squared_error(
    y_train, pred_train_no_ohe
)

print('val RMSE: ', rmse_val_no_ohe)
print('train RMSE: ', rmse_train_no_ohe)
print(
    'Gap: ',
    rmse_val_no_ohe - rmse_train_no_ohe
)

Train: (616293, 35) -> (610970, 35)
Validation: (133316, 35) -> (132190, 35)
Train 제거 행: 5323
Validation 제거 행: 1126
Learning rate set to 0.071327
0:	learn: 0.3046440	test: 0.2481225	best: 0.2481225 (0)	total: 175ms	remaining: 8m 45s
200:	learn: 0.2223323	test: 0.1940703	best: 0.1940703 (200)	total: 32s	remaining: 7m 25s
400:	learn: 0.2115546	test: 0.1905899	best: 0.1905899 (400)	total: 1m 4s	remaining: 6m 55s
600:	learn: 0.2048428	test: 0.1893610	best: 0.1893610 (600)	total: 1m 43s	remaining: 6m 54s
800:	learn: 0.1998609	test: 0.1888210	best: 0.1887844 (798)	total: 2m 20s	remaining: 6m 24s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1885875302
bestIteration = 839

Shrink model to first 840 iterations.
Best iteration: 839
Tree count: 840
Best score: 0.18858753018474964

val RMSE:  0.18858753013891283
train RMSE:  0.19815320147553897
Gap:  -0.009565671336626141


In [14]:
# ============================================================
# 4. 최종모델(15) + 데이터셋 필터링 + 필터링된 행의 복원
# ============================================================
# wl_diff_6h < -n 인 train, val 제거하고 진행
# 위 조건에 해당하는 행은 종속변수로 -wl_diff_6h을 넣음

# 필터링 전 데이터 개수
print('=== 필터링 전 ===')
print('Train:', len(train))
print('Validation:', len(val))
print()

# 필터링
train_filtered = train.loc[
    train['wl_diff_6h'] >= -6
].copy()

val_filtered = val.loc[
    val['wl_diff_6h'] >= -6
].copy()

# 필터링 후 데이터 개수
print('=== 필터링 후 ===')
print('Train:', len(train_filtered))
print('Validation:', len(val_filtered))
print('Train 제거 행:', len(train) - len(train_filtered))
print('Validation 제거 행:', len(val) - len(val_filtered))
print()
print()

# ============================================================
# 2.필터링 후, 종속변수 설정
# ============================================================

y_train = train_filtered[target_col]
y_val = val_filtered[target_col]


# station_id가 포함된 데이터셋 생성
# 기존 데이터와의 연결 차단을 위해 .copy() 사용
X_train_with_station = train_filtered[num_cols_fe + ['station_id']].copy()
X_val_with_station = val_filtered[num_cols_fe + ['station_id']].copy()

# CatBoost 전용 범주형 변수 취급: str로 자료형 변환
X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_val_with_station['station_id'] = X_val_with_station['station_id'].astype(str)

# 모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train,
    early_stopping_rounds=100,
    eval_set=(X_val_with_station, y_val),
    verbose=200
)

# ============================================================
# 4. 예측 및 RMSE 비교
# ============================================================

# 필터링된 데이터 예측
pred_val_no_ohe = model_base_no_ohe.predict(X_val_with_station)

print('Best iteration:', model_base_no_ohe.best_iteration_)
print('Tree count:', model_base_no_ohe.tree_count_)
print('Best score:', model_base_no_ohe.best_score_['validation']['RMSE'])
print()
print()

# 제외된 행에 -wl_diff_6h을 넣고자 일단 전부 다 넣은 df
pred_val_all = -val['wl_diff_6h'].copy()

# 필터링 모델링 예측 행은 다시 덮어쓰기
pred_val_all.loc[val_filtered.index] = pred_val_no_ohe

# 데이터 개수 검증
print('모두 합친 뒤 최종 Validation:', len(pred_val_all))
print()

assert len(pred_val_all) == len(val)
assert pred_val_all.index.equals(val.index)

rmse_val_no_ohe = root_mean_squared_error(
    val[target_col], pred_val_all
)

print('val RMSE: ', rmse_val_no_ohe)

=== 필터링 전 ===
Train: 616293
Validation: 133316

=== 필터링 후 ===
Train: 616170
Validation: 133293
Train 제거 행: 123
Validation 제거 행: 23


Learning rate set to 0.071422
0:	learn: 0.3803574	test: 0.3352320	best: 0.3352320 (0)	total: 184ms	remaining: 9m 10s
200:	learn: 0.2302780	test: 0.2031879	best: 0.2031879 (200)	total: 34.1s	remaining: 7m 54s
400:	learn: 0.2181291	test: 0.1974544	best: 0.1974478 (399)	total: 1m 8s	remaining: 7m 22s
600:	learn: 0.2107126	test: 0.1951323	best: 0.1951323 (600)	total: 1m 47s	remaining: 7m 9s
800:	learn: 0.2053392	test: 0.1940873	best: 0.1940784 (785)	total: 2m 19s	remaining: 6m 21s
1000:	learn: 0.2010959	test: 0.1934240	best: 0.1934240 (1000)	total: 2m 52s	remaining: 5m 44s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1933117859
bestIteration = 1065

Shrink model to first 1066 iterations.
Best iteration: 1065
Tree count: 1066
Best score: 0.1933117858839308


모두 합친 뒤 최종 Validation: 133316

val RMSE:  0.19358381327496477


In [15]:
# 분석 대상 관측소
stations = ['ST0358', 'ST0292', 'ST0326']

for name, df in [
    ('Train', train),
    ('Val', val)

]:
    for station in stations:
        data = df.loc[
            ((df['station_id'] == station) & (df['wl_diff_6h'] < -4)),
            ['timestamp', 'station_id'] + wl_cols + [target_col]
        ]

        print(f'\n{name} | {station} | {len(data):,}개')
        display(data)

#     ('Test', test)


Train | ST0358 | 311개


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000561359,2100-01-01 01:00:00,ST0358,-1.66,-0.45,1.91,2.55,-1.69,-2.12,4.30
TR_000164077,2100-01-21 07:00:00,ST0358,-2.24,-1.44,2.01,3.79,-2.17,-2.33,5.99
TR_000225662,2100-01-21 08:00:00,ST0358,-2.29,-2.24,0.34,3.98,-2.18,-2.33,6.36
TR_000368174,2100-01-21 09:00:00,ST0358,-2.29,-2.29,-1.44,3.30,-2.19,-2.41,6.36
TR_000388142,2100-01-31 09:00:00,ST0358,-1.55,-0.98,0.94,2.94,-1.66,-0.79,4.01
TR_000257547,2100-01-31 21:00:00,ST0358,-2.00,-1.27,0.62,2.46,-1.55,-1.66,5.00
TR_000026925,2100-01-31 22:00:00,ST0358,-2.27,-2.00,-0.35,2.27,-1.39,-0.81,5.60
TR_000062299,2100-02-01 09:00:00,ST0358,-1.47,-0.45,2.28,3.00,-2.00,-1.55,3.56
TR_000686641,2100-02-01 10:00:00,ST0358,-2.22,-1.47,1.00,3.33,-2.27,-1.39,4.95



Train | ST0292 | 210개


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000492897,2100-01-21 09:00:00,ST0292,-2.72,-2.02,-0.28,2.08,-2.81,-3.02,4.89
TR_000556269,2100-01-31 23:00:00,ST0292,-3.33,-2.82,-1.53,0.71,-2.89,-2.68,4.87
TR_000058289,2100-02-01 11:00:00,ST0292,-2.54,-1.91,-0.29,1.54,-3.33,-2.89,3.12
TR_000702356,2100-02-01 12:00:00,ST0292,-3.07,-2.54,-1.14,1.44,-3.57,-2.62,4.10
TR_000673169,2100-02-20 00:00:00,ST0292,-3.03,-2.42,-0.78,2.30,-3.59,-3.32,4.75
TR_000054371,2100-02-20 01:00:00,ST0292,-3.08,-3.03,-1.67,1.31,-3.17,-2.84,4.80
TR_000100855,2100-03-02 07:00:00,ST0292,-2.58,-1.88,-0.17,1.79,-3.23,-2.93,3.52
TR_000040190,2100-03-02 08:00:00,ST0292,-3.13,-2.58,-1.09,1.61,-3.71,-3.40,4.07
TR_000718422,2100-04-01 00:00:00,ST0292,-2.85,-2.28,-0.79,1.99,-3.37,-3.22,4.27



Train | ST0326 | 198개


...

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000749200,2103-06-15 00:00:00,ST0358,-2.64,-2.65,-1.73,3.32,-2.72,-2.64,6.84
TR_000263461,2103-06-15 01:00:00,ST0358,-2.10,-2.64,-2.65,1.97,-1.70,-1.00,6.30
TR_000596538,2103-06-25 03:00:00,ST0358,-2.19,-0.40,3.01,2.61,-1.45,-2.86,4.82
TR_000457634,2103-06-25 04:00:00,ST0358,-2.85,-2.19,1.53,3.87,-2.94,-2.86,7.38
TR_000293983,2103-07-05 01:00:00,ST0358,-1.95,-0.85,1.67,2.61,-1.98,-2.49,4.24
TR_000121400,2103-07-05 02:00:00,ST0358,-2.69,-1.95,0.33,2.82,-2.28,-2.36,5.80
TR_000082870,2103-07-05 03:00:00,ST0358,-2.65,-2.69,-0.85,2.42,-1.79,-1.56,5.87
TR_000339809,2103-07-15 05:00:00,ST0358,-0.56,0.99,4.58,3.85,-1.77,-1.36,2.53
TR_000568701,2103-07-15 06:00:00,ST0358,-1.99,-0.56,3.11,4.81,-2.04,-2.02,5.85



Val | ST0292 | 42개


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000334066,2103-06-15 00:00:00,ST0292,-2.86,-2.21,-0.44,2.34,-2.86,-3.14,5.09
TR_000555520,2103-06-15 01:00:00,ST0292,-3.43,-2.86,-1.41,1.80,-3.40,-3.62,5.66
TR_000395312,2103-06-15 02:00:00,ST0292,-3.33,-3.43,-2.21,0.69,-2.95,-2.72,5.56
TR_000135563,2103-07-05 03:00:00,ST0292,-2.80,-2.15,-0.56,1.35,-2.97,-3.40,3.93
TR_000464849,2103-07-15 08:00:00,ST0292,-2.12,-1.50,0.42,3.45,-2.95,-2.69,4.60
TR_000040410,2103-07-15 09:00:00,ST0292,-2.65,-2.12,-0.70,3.06,-3.47,-3.14,5.13
TR_000095272,2103-07-15 10:00:00,ST0292,-2.95,-2.65,-1.50,1.64,-2.75,-2.27,5.43
TR_000006962,2103-07-15 20:00:00,ST0292,-2.32,-1.71,0.13,2.48,-2.12,-2.95,5.52
TR_000665206,2103-07-15 21:00:00,ST0292,-2.85,-2.32,-0.91,2.35,-2.65,-3.47,6.39



Val | ST0326 | 46개


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000466466,2103-06-25 02:00:00,ST0326,-2.46,-1.51,1.41,2.59,-1.74,-3.27,5.37
TR_000094091,2103-06-25 03:00:00,ST0326,-3.43,-2.46,-0.09,2.90,-2.69,-3.79,7.20
TR_000666483,2103-06-25 04:00:00,ST0326,-3.84,-3.43,-1.51,2.53,-3.02,-3.31,7.71
TR_000451727,2103-07-05 01:00:00,ST0326,-2.40,-1.81,-0.42,2.05,-1.87,-1.83,4.83
TR_000578936,2103-07-15 04:00:00,ST0326,-1.45,-0.39,2.87,3.25,-2.64,-2.03,3.59
TR_000187929,2103-07-15 05:00:00,ST0326,-2.53,-1.45,1.46,3.67,-3.40,-2.95,5.37
TR_000150178,2103-07-15 06:00:00,ST0326,-3.23,-2.53,-0.39,3.63,-3.63,-3.01,6.07
TR_000394852,2103-07-15 07:00:00,ST0326,-2.75,-3.23,-1.45,2.87,-2.60,-1.67,5.59
TR_000284775,2103-07-15 16:00:00,ST0326,-2.11,-0.33,1.92,2.14,-1.45,-2.64,4.46


In [17]:
# wl_diff_6h = 종속변수인 행들은 혹시 조수간만의 차에 의한건가?

# 1. 조건에 해당하는 행 추출
tidal_pattern = train.loc[
    (train['wl_diff_6h'] < -1) &
    ((train['target_maxrise_6h'] + train['wl_diff_6h']).abs() < 0.5)
]

# 2. 관측소별 조건 만족 건수 및 전체 데이터 대비 비율
tidal_station = pd.DataFrame({
    'total_count': train.groupby('station_id').size(),
    'pattern_count': tidal_pattern.groupby('station_id').size()
})

tidal_station['pattern_count'] = tidal_station['pattern_count'].fillna(0).astype(int)

tidal_station['pattern_ratio'] = (
    tidal_station['pattern_count'] / tidal_station['total_count'] * 100
)

# 3. 조건 만족 건수 기준 내림차순 정렬
tidal_station = tidal_station.sort_values(
    'pattern_count', ascending=False
)

display(tidal_station)

,total_count,pattern_count,pattern_ratio
station_id,,,
ST0110,1379,272,19.724438
ST0326,1379,260,18.854242
ST0159,1379,220,15.953590
ST0358,1379,213,15.445975
ST0347,1379,208,15.083394
ST0415,989,201,20.323559
ST0292,1354,200,14.771049
ST0313,1379,124,8.992023
ST0234,1068,106,9.925094


In [18]:
# 조석 주기성 후보 관측소 9개만 별도로 학습시키기

# 조석 주기성 후보
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# Train 분리
tidal_train = train.loc[train['station_id'].isin(tidal_stations)].copy()
normal_train = train.loc[~train['station_id'].isin(tidal_stations)].copy()

# Validation 분리
tidal_val = val.loc[val['station_id'].isin(tidal_stations)].copy()
normal_val = val.loc[~val['station_id'].isin(tidal_stations)].copy()

# 분리 결과 확인
print(f"Train : {len(train):,}")
print(f"  Tidal  : {len(tidal_train):,}")
print(f"  Normal : {len(normal_train):,}")

print(f"\nValidation : {len(val):,}")
print(f"  Tidal  : {len(tidal_val):,}")
print(f"  Normal : {len(normal_val):,}")

Train : 616,293
  Tidal  : 11,685
  Normal : 604,608

Validation : 133,316
  Tidal  : 2,566
  Normal : 130,750


In [22]:
# ============================================================
# 조석 후보 9개 관측소 별도 CatBoost 회귀모델
# ============================================================

# 1. 독립변수, 종속변수 분리
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_val = tidal_val[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train['target_maxrise_6h']
y_tidal_val = tidal_val['target_maxrise_6h']

# 2. CatBoost 전용 범주형 변수 처리
X_tidal_train['station_id'] = X_tidal_train['station_id'].astype(str)
X_tidal_val['station_id'] = X_tidal_val['station_id'].astype(str)

# 3. 모델 정의
model_tidal = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 4. 모델 학습
model_tidal.fit(
    X_tidal_train,
    y_tidal_train,
    early_stopping_rounds=100,
    eval_set=(X_tidal_val, y_tidal_val),
    verbose=200
)

# 5. 예측
pred_tidal_val = model_tidal.predict(X_tidal_val)
pred_tidal_train = model_tidal.predict(X_tidal_train)

# 6. 학습 결과 확인
print('Best iteration:', model_tidal.best_iteration_)
print('Tree count:', model_tidal.tree_count_)
print('Best score:', model_tidal.best_score_['validation']['RMSE'])
print()

# 7. RMSE 계산
rmse_tidal_val = root_mean_squared_error(
    y_tidal_val, pred_tidal_val
)

rmse_tidal_train = root_mean_squared_error(
    y_tidal_train, pred_tidal_train
)

print('val RMSE: ', rmse_tidal_val)
print('train RMSE: ', rmse_tidal_train)
print('Gap: ', rmse_tidal_val - rmse_tidal_train)

Learning rate set to 0.038324
0:	learn: 1.7155655	test: 1.6789126	best: 1.6789126 (0)	total: 173ms	remaining: 8m 38s
200:	learn: 0.4074975	test: 0.4301416	best: 0.4301416 (200)	total: 8.73s	remaining: 2m 1s
400:	learn: 0.3449974	test: 0.3856054	best: 0.3856054 (400)	total: 17.1s	remaining: 1m 50s
600:	learn: 0.3077946	test: 0.3683486	best: 0.3683486 (600)	total: 25.7s	remaining: 1m 42s
800:	learn: 0.2828673	test: 0.3595638	best: 0.3595105 (799)	total: 33.9s	remaining: 1m 33s
1000:	learn: 0.2631933	test: 0.3543644	best: 0.3543644 (1000)	total: 42s	remaining: 1m 23s
1200:	learn: 0.2471664	test: 0.3504691	best: 0.3504691 (1200)	total: 50.2s	remaining: 1m 15s
1400:	learn: 0.2333971	test: 0.3487546	best: 0.3486535 (1395)	total: 58.3s	remaining: 1m 6s
1600:	learn: 0.2214358	test: 0.3474502	best: 0.3473743 (1592)	total: 1m 6s	remaining: 58.2s
1800:	learn: 0.2109443	test: 0.3464440	best: 0.3463395 (1794)	total: 1m 15s	remaining: 50s
2000:	learn: 0.2017917	test: 0.3456331	best: 0.3456080 (1997)

In [24]:
# 1. 기존 통합모델 독립변수 및 종속변수 정의
X_train_with_station = train[num_cols_fe + ['station_id']].copy()
X_val_with_station = val[num_cols_fe + ['station_id']].copy()

y_train = train['target_maxrise_6h']
y_val = val['target_maxrise_6h']

X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_val_with_station['station_id'] = X_val_with_station['station_id'].astype(str)

# 2. 기존 통합모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 3. 모델 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train,
    early_stopping_rounds=100,
    eval_set=(X_val_with_station, y_val),
    verbose=200
)

# 4. 기존 통합모델로 Tidal Validation 예측
pred_tidal_val_base = model_base_no_ohe.predict(X_tidal_val)

# 5. RMSE 비교
rmse_tidal_val_base = root_mean_squared_error(
    y_tidal_val, pred_tidal_val_base
)

print('기존 통합모델 Tidal RMSE:', rmse_tidal_val_base)
print('별도 Tidal 모델 RMSE:', rmse_tidal_val)
print('RMSE 변화:', rmse_tidal_val - rmse_tidal_val_base)

Learning rate set to 0.071425
0:	learn: 0.3895386	test: 0.3444227	best: 0.3444227 (0)	total: 169ms	remaining: 8m 26s
200:	learn: 0.2300278	test: 0.2035631	best: 0.2035631 (200)	total: 32.7s	remaining: 7m 35s
400:	learn: 0.2182347	test: 0.1979222	best: 0.1979222 (400)	total: 1m 6s	remaining: 7m 11s
600:	learn: 0.2110872	test: 0.1958141	best: 0.1958141 (600)	total: 1m 38s	remaining: 6m 31s
800:	learn: 0.2055799	test: 0.1947004	best: 0.1947004 (800)	total: 2m 12s	remaining: 6m 3s
1000:	learn: 0.2011655	test: 0.1941687	best: 0.1941687 (1000)	total: 2m 43s	remaining: 5m 27s
1200:	learn: 0.1974401	test: 0.1937451	best: 0.1937451 (1200)	total: 3m 15s	remaining: 4m 52s
1400:	learn: 0.1941079	test: 0.1933177	best: 0.1932767 (1372)	total: 3m 47s	remaining: 4m 19s
1600:	learn: 0.1913972	test: 0.1931965	best: 0.1931434 (1536)	total: 4m 19s	remaining: 3m 47s
1800:	learn: 0.1888074	test: 0.1930250	best: 0.1929570 (1778)	total: 4m 52s	remaining: 3m 14s
Stopped by overfitting detector  (100 iterations

In [25]:
# ============================================================
# Normal 관측소 전용 CatBoost 회귀모델
# ============================================================

# 1. 독립변수, 종속변수 분리
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_val = normal_val[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train['target_maxrise_6h']
y_normal_val = normal_val['target_maxrise_6h']

# 2. CatBoost 전용 범주형 변수 처리
X_normal_train['station_id'] = X_normal_train['station_id'].astype(str)
X_normal_val['station_id'] = X_normal_val['station_id'].astype(str)

# 3. 모델 정의
model_normal = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 4. 모델 학습
model_normal.fit(
    X_normal_train,
    y_normal_train,
    early_stopping_rounds=100,
    eval_set=(X_normal_val, y_normal_val),
    verbose=200
)

# 5. 예측
pred_normal_val = model_normal.predict(X_normal_val)
pred_normal_train = model_normal.predict(X_normal_train)

# 6. 학습 결과 확인
print('Best iteration:', model_normal.best_iteration_)
print('Tree count:', model_normal.tree_count_)
print('Best score:', model_normal.best_score_['validation']['RMSE'])
print()

# 7. RMSE 계산
rmse_normal_val = root_mean_squared_error(
    y_normal_val, pred_normal_val
)

rmse_normal_train = root_mean_squared_error(
    y_normal_train, pred_normal_train
)

print('val RMSE: ', rmse_normal_val)
print('train RMSE: ', rmse_normal_train)
print('Gap: ', rmse_normal_val - rmse_normal_train)

Learning rate set to 0.07121
0:	learn: 0.2846285	test: 0.2243682	best: 0.2243682 (0)	total: 242ms	remaining: 12m 6s
200:	learn: 0.2179375	test: 0.1896694	best: 0.1896694 (200)	total: 34.2s	remaining: 7m 56s
400:	learn: 0.2078211	test: 0.1870848	best: 0.1870649 (399)	total: 1m 6s	remaining: 7m 9s
600:	learn: 0.2017569	test: 0.1867552	best: 0.1867185 (590)	total: 1m 43s	remaining: 6m 53s
800:	learn: 0.1969781	test: 0.1866371	best: 0.1865962 (742)	total: 2m 23s	remaining: 6m 34s
1000:	learn: 0.1929121	test: 0.1864860	best: 0.1864503 (952)	total: 2m 56s	remaining: 5m 51s
1200:	learn: 0.1895492	test: 0.1862929	best: 0.1862671 (1196)	total: 3m 27s	remaining: 5m 10s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.186251702
bestIteration = 1211

Shrink model to first 1212 iterations.
Best iteration: 1211
Tree count: 1212
Best score: 0.18625170199354416

val RMSE:  0.1862517019446085
train RMSE:  0.18867773787758255
Gap:  -0.002426035932974052


In [27]:
# ============================================================
# Tidal + Normal 예측값 결합 및 전체 Validation RMSE
# ============================================================

# 1. 두 모델의 예측값 결합 (기존 인덱스 유지)
pred_val_split = pd.Series(index=val.index, dtype=float)

pred_val_split.loc[tidal_val.index] = pred_tidal_val
pred_val_split.loc[normal_val.index] = pred_normal_val

# 2. 전체 Validation RMSE 계산
rmse_val_split = root_mean_squared_error(
    val['target_maxrise_6h'],
    pred_val_split
)

print(f'분리모델 전체 Validation RMSE: {rmse_val_split:.6f}')

분리모델 전체 Validation RMSE: 0.190577
